# Semaine 10 — Jour 1 : Classification de textes et protocole sans fuite

Entraîner réellement un classifieur TF-IDF et une tête linéaire sur le train.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J1_classification.ipynb` et `notebooks/week10/teacher/S10_J1_classification_teacher.ipynb`.

Critère de réussite : Interpréter macro-F1, matrice de confusion, couverture et abstention.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Entraîner réellement un classifieur TF-IDF et une tête linéaire sur le train.
- Séparer train, adaptation, validation et test avec contrôle des groupes.
- Interpréter macro-F1, matrice de confusion, couverture et abstention.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J1 — Classification de textes et protocole sans fuite


## 1. Définir la tâche avant le modèle
Le projet classe des tickets en acces, deploiement ou performance. Le texte est l’entrée; la catégorie attendue est l’étiquette. Une classification mono-étiquette suppose qu’une seule catégorie suffit. Un ticket qui mêle release lente et jeton expiré peut contredire cette hypothèse. Documenter la règle d’annotation et prévoir une revue plutôt que présenter la classe comme une vérité universelle.

Le corpus contient 72 tickets fictifs : 36 train, 12 adaptation, 12 validation et 12 test. Les classes sont équilibrées dans chaque partition. Ces textes servent à entraîner et inspecter le protocole; ils ne représentent pas la diversité d’incidents réels. Les résultats faciles sur des formulations proches doivent rester visibles comme limite.

## 2. Empêcher les fuites
Les lignes possèdent id, group_id, split, label et text. Le validateur refuse les identifiants dupliqués, textes dupliqués normalisés et groupes répartis sur plusieurs splits. En pratique, grouper les messages d’un même incident, client ou conversation avant la répartition. Un simple tirage de lignes peut envoyer une reformulation du même problème dans le test.

Le vocabulaire et les IDF sont appris seulement sur train. Adaptation et validation utilisent transform, jamais fit. La validation sélectionne les réglages; le test sert à l’évaluation après sélection. Réexaminer le test pour choisir une nouvelle variante le transforme progressivement en jeu de développement.

## 3. Représenter et apprendre
TfidfVectorizer produit des unigrammes et bigrammes, normalise les accents et la casse, puis applique TF-IDF et une norme L2. La matrice creuse évite de stocker tous les zéros. Les bigrammes conservent un peu de contexte local, sans résoudre l’ordre global, les négations complexes ou les paraphrases éloignées.

SGDClassifier avec log_loss apprend réellement une tête linéaire par descente de gradient stochastique. Il ne s’agit ni d’un appel LLM ni de prédictions préfabriquées. Dans ce schéma multiclasses, scikit-learn combine des classifieurs one-vs-rest. La probabilité maximale n’est pas calibrée pour les exigences du métier.

## 4. Lire les métriques
La matrice de confusion utilise un ordre de classes explicite. Le macro-F1 donne le même poids à chaque classe; l’accuracy donne le même poids à chaque observation. Avec peu d’exemples et un corpus équilibré très simple, un score parfait ne prouve pas une fiabilité élevée sur de nouveaux tickets.

Le seuil d’acceptation est choisi sur validation parmi cinq valeurs fixées. La règle cherche la couverture maximale sous une accuracy sélective illustrative d’au moins 0,9. Si aucun seuil candidat ne satisfait la contrainte, la sélection est refusée plutôt que de conserver un seuil arbitraire. Coverage est la part de tickets acceptés; selective_accuracy porte uniquement sur ces tickets. En l’absence d’acceptation, ce dernier indicateur vaut null, pas un succès parfait.

## 5. Hors distribution
Le classifieur peut être confiant sur une phrase étrangère au domaine, notamment si quelques mots ordinaires sont connus du vocabulaire. Le garde de vecteur nul rejette seulement une entrée sans aucune caractéristique connue. Il ne constitue pas un détecteur fiable de hors-distribution.

Les exemples ood sont conservés sans inventer d’étiquette gold parmi les trois catégories. Ils illustrent le comportement du modèle et ses limites. Une exploitation réelle demanderait des négatifs représentatifs, une calibration, un contrôle des changements de distribution et une politique de revue.

## À retenir
L’apprentissage commence par le contrat de tâche et la séparation des données. Un bon score n’efface ni les ambiguïtés d’étiquette ni la faiblesse du corpus.


## Diagramme du jour

```mermaid
flowchart TD
 A["Dataset et groupes"] --> B["Train : vocabulaire et tête"]
 A --> C["Adaptation : poids seuls"]
 A --> D["Validation : sélection"]
 B --> C
 C --> D
 D --> E["Test après sélection"]
 E --> F["Métriques et limites"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import trained, read_data, subset, classification_metrics
bundle = trained()
rows = read_data("tickets.json")
report = classification_metrics(bundle["vectorizer"], bundle["baseline"], subset(rows, "test"), bundle["baseline_threshold"])
assert report["count"] == 12 and len(report["confusion_matrix"]) == 3
print({k: v for k, v in report.items() if k != "cases"})

# Lab J1 — Classification de textes et protocole sans fuite

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day01/labs/run.py
python3 -m examples.week10.demo --day 1
```

Résultat attendu : **évaluation sur 12 tickets test et comportements hors distribution conservés**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Tracer les partitions

Qui peut servir à fit du vocabulaire, adaptation des poids, sélection d’epochs et mesure finale ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Interpréter les scores

Pourquoi afficher couverture et accuracy sélective ensemble ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Tester le hors-domaine

Pourquoi une probabilité élevée sur un ticket inconnu peut-elle rester trompeuse ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Comment éviter une fuite par conversation ?

## Question 2

Pourquoi une baseline linéaire avant un Transformer ?


# Challenge

Prouver qu’un terme présent seulement dans le test n’entre pas dans le vocabulaire. Créer aussi un group_id partagé entre train et test et vérifier le rejet du dataset.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Corrigé des exercices

## Exercice 1

Train pour vocabulaire et tête initiale; adapt pour poursuivre les poids; validation pour réglages/epochs/seuil; test pour la mesure après sélection. Ne pas entraîner l’IDF sur les quatre ensembles.

## Exercice 2

Un système peut refuser presque tout et être parfait sur quelques cas simples acceptés. Le dénominateur change; null sur aucun accepté doit rester distinct de 1.

## Exercice 3

La tête répartit les scores entre les classes connues, même si aucune n’est appropriée. Une représentation non nulle ne prouve pas que le texte est dans le domaine.


# Réponses d’entretien

## Question 1

Définir un group_id au niveau de la conversation ou de l’incident et maintenir chaque groupe dans une seule partition. Examiner aussi les doublons et quasi-doublons.

## Question 2

Elle est rapide, interprétable et donne une référence mesurée. Un modèle plus coûteux doit résoudre une erreur identifiée et être comparé sous le même protocole.


# Solution du challenge

Modifier une copie du texte test en y ajoutant un mot unique, réentraîner avec train_bundle puis vérifier son absence de vocabulary_. Les données de sélection ne changent pas. Pour le groupe partagé, modifier un group_id test avec celui du train : validate_dataset lève ContractError avant tout entraînement.

## Vérification exécutable

Exécuter après les cellules du lab élève.

In [ ]:
from copy import deepcopy
from examples.week10.nlp import validate_dataset, ContractError
changed = deepcopy(rows)
first_train = next(r for r in changed if r["split"] == "train")
first_test = next(r for r in changed if r["split"] == "test")
first_test["group_id"] = first_train["group_id"]
try:
    validate_dataset(changed)
except ContractError:
    print("Fuite de groupe refusée")
else:
    raise AssertionError("Groupe partagé accepté")
from examples.week10.nlp import train_bundle
reserved = deepcopy(rows)
for row in reserved:
    if row["split"] == "test":
        row["text"] += " termeuniquedutest"
retrained = train_bundle(reserved)
assert "termeuniquedutest" not in retrained["vectorizer"].vocabulary_
assert retrained["vectorizer"].vocabulary_ == bundle["vectorizer"].vocabulary_

# Revue pédagogique — J1

## Acquis

- Entraîner réellement un classifieur TF-IDF et une tête linéaire sur le train.
- Séparer train, adaptation, validation et test avec contrôle des groupes.
- Interpréter macro-F1, matrice de confusion, couverture et abstention.

## Barème indicatif sur 10

- 3 : concepts reliés au contrat métier.
- 3 : reproduction et interprétation du lab.
- 2 : exercices justifiés.
- 2 : challenge et limite de mesure.

Le parcours essentiel peut être terminé sans le challenge. Une confusion entre score logiciel et qualité générale, tête linéaire et LoRA, mention et assertion, ou masquage et anonymisation doit être corrigée avant la revue finale. Vérifier en particulier : La tête répartit les scores entre les classes connues, même si aucune n’est appropriée. Une représentation non nulle ne prouve pas que le texte est dans le domaine.


# Références — Jour 1

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
